# Departures without NM flight data

About 1 % of the departures have no `FLIGHT_ID_mvt`: the movement could not be matched to an NM flight, so all
`_flt` columns (LOBT, EOBT, AOBT, operator, …) are missing. For these flights neither the NM times nor the LOBT window
are available. The questions: where and when do they occur, what can still be recovered for them, and how do their
labels behave?

In [1]:
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "pyproject.toml").is_file())
FEATURES = ROOT / "data/03_features"

In [2]:
sets = {"training 2025": "train_DEP_merged_sorted.parquet", "ranking": "ranking.parquet", "final ranking": "final_ranking.parquet"}
frames = {}
for label, name in sets.items():
    f = pd.read_parquet(FEATURES / name, columns=["PHASE_mvt", "ADEP_mvt", "MVT_TIME_UTC_mvt", "SCHED_TIME_UTC_mvt",
                                                  "FLIGHT_ID_mvt", "AIRCRAFT_OPERATOR_flt", "adsb_movement_to_mvt_sec"])
    f = f.loc[f["PHASE_mvt"].eq("DEP")]
    if label == "training 2025":
        f = f.loc[pd.to_datetime(f["SCHED_TIME_UTC_mvt"], utc=True).dt.year.eq(2025)]
    f["month"] = pd.to_datetime(f["MVT_TIME_UTC_mvt"], utc=True).dt.month
    f["no NM"] = f["FLIGHT_ID_mvt"].isna()
    frames[label] = f
pd.DataFrame({label: f.groupby("month")["no NM"].mean().mul(100).round(2) for label, f in frames.items()})

,training 2025,ranking,final ranking
month,,,
1,0.93,1.60,1.60
2,0.77,NaN,0.91
3,0.66,NaN,NaN
4,0.62,NaN,NaN
5,0.87,NaN,NaN
6,1.36,NaN,1.35
7,2.06,1.48,1.48
8,1.22,NaN,NaN
9,1.34,NaN,NaN


### Per airport

In [3]:
pd.DataFrame({label: f.groupby("ADEP_mvt")["no NM"].mean().mul(100).round(2) for label, f in frames.items()})

,training 2025,ranking,final ranking
ADEP_mvt,,,
EDDF,0.85,1.16,1.08
EDDM,0.60,0.92,0.88
EGLL,0.59,0.62,0.61
EHAM,1.65,3.16,2.50
LEBL,1.03,1.47,1.27
LEMD,0.44,0.64,0.60
LFPG,1.57,1.84,1.68
LIRF,0.93,1.42,1.19
LSZH,1.70,2.85,2.33


### What is still known for them

The operator can be recovered from the flight number (`02_clean`), and ADS-B does not depend on the NM data
(`04_adsb_movement`).

In [4]:
pd.DataFrame({label: {"departures without NM": int(f["no NM"].sum()),
                      "operator known": f.loc[f["no NM"], "AIRCRAFT_OPERATOR_flt"].notna().mean(),
                      "ADS-B movement before take-off": f.loc[f["no NM"], "adsb_movement_to_mvt_sec"].gt(0).mean(),
                      "the same for flights with NM": f.loc[~f["no NM"], "adsb_movement_to_mvt_sec"].gt(0).mean()}
              for label, f in frames.items()}).round(3)

,training 2025,ranking,final ranking
departures without NM,22424.000,5283.000,9057.000
operator known,0.778,0.666,0.645
ADS-B movement before take-off,0.350,0.395,0.386
the same for flights with NM,0.536,0.573,0.572


### Artefacts without NM

Share of artefact labels among delayed departures (delay above 2000 s) without NM flight data.

In [5]:
t = pd.read_parquet(FEATURES / "train_DEP_merged_sorted.parquet",
                    columns=["ADEP_mvt", "SCHED_TIME_UTC_mvt", "FLIGHT_ID_mvt", "TAXITIME_SEC_mvt", "delay_sec"])
t = t.loc[pd.to_datetime(t["SCHED_TIME_UTC_mvt"], utc=True).dt.year.eq(2025) & t["FLIGHT_ID_mvt"].isna() & (t["delay_sec"] > 2000)]
t["artefact"] = (t["TAXITIME_SEC_mvt"] - t["delay_sec"]).abs() <= 60
t.groupby("ADEP_mvt")["artefact"].agg(["size", "mean"]).round(3)

,size,mean
ADEP_mvt,,
EDDF,1814,0.001
EDDM,972,0.001
EGLL,1410,0.003
EHAM,2422,0.000
LEBL,1719,0.001
LEMD,933,0.000
LFPG,3541,0.001
LIRF,1407,0.488
LSZH,1156,0.000


### What follows for the models

- The share is of the same order in the ranking files and higher than in 2025 at every airport, so these flights
  matter at least as much there.
- `02_clean` fills the operator from the flight number's 2025 history, and the ADS-B processing must not require an
  NM flight ID: these flights are processed like all others. ADS-B then shows a ground movement before the take-off
  for about a third of them, somewhat less often than for flights with NM data.
- Without NM data a delayed flight at LIRF is an artefact about half of the time, elsewhere almost never. The
  classifier therefore gets the airport and the stand features that work without NM times, and LIRF needs special
  care (the one rule in `09_corrector` concerns LIRF flights without NM off-block).